In [ ]:

# List all the files in a folder
import os
import random
from typing import List, Tuple

def split_into_blocks(csv_path):
    blocks = []
    current_block = []

    with open(csv_path, "r", encoding="utf-8") as f:
        for line in f:
            # Strip newline but keep commas structure
            stripped = line.rstrip("\n")
            if not stripped.strip(): 
                # Skip completely empty lines
                continue

            # Try to detect the start of a new block
            parts = stripped.split(",")
            first_cell = parts[0].strip()

            # A block starts when first cell is an integer game number (1,2,...)
            if first_cell.isdigit():
                # If there is an existing block, save it
                if current_block:
                    blocks.append(current_block)
                current_block = [stripped]  # Start new block
            else:
                # Inside a block → append line
                if current_block is not None:
                    current_block.append(stripped)

        # Append last block if exists
        if current_block:
            blocks.append(current_block)

    return blocks
        
def parse_line(line):
    parts = line.split(",")
    value = int(parts[9].strip())
    assigment = parts[10].strip()
    return value, assigment

def compute_value(assignments_player1:List[Tuple[int, str]], assignments_player2:List[Tuple[int, str]]) -> Tuple[float, float]:
    assert len(assignments_player1) == len(assignments_player2)
    v_player1, v_player2 = 0.0, 0.0
    for a1, a2 in zip(assignments_player1, assignments_player2):
        if a1[1] != a2[1]:
            v_player1 -= 0.2 * a1[0]
            v_player2 -= 0.2 * a1[0]
        elif a1[1] == a2[1] and a1[1] == "r1":
            v_player1 += a1[0]
        elif a1[1] == a2[1] and a1[1] == "r2":
            v_player2 += a1[0]

    return (v_player1, v_player2)

def signature(csv_file):
    blocks = split_into_blocks(csv_file)[1:]
    signature = ""
    for b in blocks:
        for row in b[3:]:
            row_split = row.split(',')
            x, y = row_split[-4], row_split[-3]
            signature += f"{x},{y};"
    return signature

folder_path = "./data/Dor-humans/bargaining-table/"
files = os.listdir(folder_path)

# Filter files with only results
PlayerNmber1_files, PlayerNmber2_files = [], []
for f in files:
    if "BT" in f:
        if "PlayerNmber1" in f:
            PlayerNmber1_files.append(f)
        if "PlayerNmber2" in f:
            PlayerNmber2_files.append(f)

game_num = 2
num_samples = 100
count = 0
player1_payoff, player2_payoff = 0.0, 0.0
for f1 in PlayerNmber1_files:
    sign_f1 = signature(os.path.join(folder_path, f1))
    sign_f2 = ""
    PlayerNmber2_files_copy = PlayerNmber2_files.copy()
    for f2 in PlayerNmber2_files_copy:
        sign_f2 = signature(os.path.join(folder_path, f2))
        if sign_f1 != sign_f2:
            continue
        
        blocks1 = split_into_blocks(os.path.join(folder_path, f1))[game_num][3:]
        blocks2 = split_into_blocks(os.path.join(folder_path, f2))[game_num][3:]
        
        # print(os.path.join(folder_path, f1))
        # print(os.path.join(folder_path, f2))
        
        for r1,r2 in zip(blocks1, blocks2):
            r1_v = r1.split(',')
            r2_v = r2.split(',')
            r1_t = r1_v[-1].strip()
            r2_t = r2_v[-1].strip()
            if r1_t == r2_t and r1_t == "r1":
                player1_payoff += int(r1_v[-2].strip())
            elif r1_t == r2_t and r1_t == "r2":
                player2_payoff += int(r2_v[-2].strip())
            else:
                player1_payoff -= 0.2 * int(r1_v[-2].strip())
                player2_payoff -= 0.2 * int(r2_v[-2].strip())
                
        count += 1



In [ ]:
print(player1_payoff / count)
print(player2_payoff / count)
print(count)